In [ ]:
import soundfile as sf
import io
import numpy as np
import base64
import requests
import subprocess
import librosa
from IPython.display import Audio

In [ ]:
SAMPLE_RATE = 16000
# URL = "https://api.sangjeong.com:8080/whisper/stt_duration"
URL = "http://localhost:8000/asr/stt_duration"

In [ ]:
def compress_to_opus(bytes: bytes) -> bytes:
  process = subprocess.Popen(
    ["ffmpeg", "-i", "pipe:0", "-c:a", "libopus", "-f", "opus", "pipe:1"],
    stdin=subprocess.PIPE,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE # subprocess.DEVNULL 하면 속도 조금 더 빨라짐
  )

  out, err = process.communicate(input=bytes)
  return out, err 

In [ ]:
def np_to_wav(audio: np.ndarray, sample_rate:int) -> bytes:
  buffer = io.BytesIO()
  sf.write(buffer, audio, sample_rate, format='wav')
  return buffer.getvalue()

In [ ]:
audio, sr = librosa.load(".data/news_with_english.mp3", sr=SAMPLE_RATE)

In [ ]:
total_samples = len(audio)

segments = []
pos = 0
while pos < total_samples:
  rand_len = int(np.random.normal(loc=4000, scale=400))
  rand_len = np.clip(rand_len, 3500, 4500)
  end = min(pos + rand_len, total_samples)

  chunk = audio[pos:end]
  segments.append(chunk)
  pos = end

In [ ]:
idx = 0

In [ ]:
audio = segments[idx]
idx += 1
Audio(audio, rate=SAMPLE_RATE)

In [ ]:
output = []
for segment in segments:
  audio = np_to_wav(segment, SAMPLE_RATE)
  # audio, _ = compress_to_opus(audio)
  audio_b64 = base64.b64encode(audio).decode('utf-8')
  params = {
    "group": "4", 
    "user": "1", 
    "audio": audio_b64
  }

  res = requests.get(URL, params=params)
  # print(res)
  d = res.json()
  print(d)
  output = [ *output, *d['completed'] ]

In [ ]:
for s in output:
  print(s["text"])
  print(f"\t start:{s['words'][0]['start']}, end:{s['words'][-1]['end']}")